# Graph Grammars

This notebook demonstrates graph grammar concepts using topologic_fast.

## What are Graph Grammars?

Graph grammars are formal systems for generating and transforming graphs through rewrite rules.
Each rule specifies:
- A **left-hand side (LHS)**: A pattern to match in the graph
- A **right-hand side (RHS)**: The replacement pattern

In architectural design, graph grammars can be used to:
- Generate floor plans
- Apply design patterns
- Transform spatial layouts
- Encode design rules

## Available Features in topologic_fast

- `Wire.IsSimilar()` - Compare wire shapes (angle-based matching)
- Manual rule application through cell splitting and transformation
- `CellComplex.ByCells()` - Combine generated cells
- `Graph.ByTopology()` - Create connectivity graphs from layouts

## Features Not Yet Implemented

- `Topology.IsSimilar()` - General topology comparison with transformation matrix
- `ShapeGrammar` class - Full grammar rule management system

In [ ]:
import topologic_fast as tf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import math

## 1. Representing Grammar Rules

We'll represent grammar rules using edges where:
- The start vertex stores the LHS shape (as a Wire)
- The end vertex stores the RHS shape (as a Wire)
- The edge stores metadata about the rule

In [ ]:
class GrammarRule:
    """Represents a graph grammar rule with LHS and RHS patterns."""
    
    def __init__(self, name, lhs_wire, rhs_wire, description=""):
        self.name = name
        self.description = description
        self.lhs_wire = lhs_wire  # Left-hand side pattern
        self.rhs_wire = rhs_wire  # Right-hand side pattern
        
        # Create vertices at the centroids of the shapes
        lhs_center = self._wire_centroid(lhs_wire)
        rhs_center = self._wire_centroid(rhs_wire)
        
        # Offset RHS to the right for visualization
        self.lhs_vertex = tf.Vertex.ByCoordinates(lhs_center[0] - 3, lhs_center[1], lhs_center[2])
        self.rhs_vertex = tf.Vertex.ByCoordinates(rhs_center[0] + 3, rhs_center[1], rhs_center[2])
        
        # Create edge representing the rule
        self.rule_edge = tf.Edge.ByStartVertexEndVertex(self.lhs_vertex, self.rhs_vertex)
    
    def _wire_centroid(self, wire):
        """Calculate the centroid of a wire."""
        vertices = wire.Vertices()
        if len(vertices) == 0:
            return (0, 0, 0)
        x = sum(v.X() for v in vertices) / len(vertices)
        y = sum(v.Y() for v in vertices) / len(vertices)
        z = sum(v.Z() for v in vertices) / len(vertices)
        return (x, y, z)
    
    def __repr__(self):
        return f"GrammarRule({self.name}: {self.description})"


print("GrammarRule class defined.")

## 2. Create Shape Patterns

Define various shapes that will be used in grammar rules.

In [ ]:
def create_rectangle(width, length, cx=0, cy=0):
    """Create a rectangular wire centered at (cx, cy)."""
    hw, hl = width/2, length/2
    v1 = tf.Vertex.ByCoordinates(cx - hw, cy - hl, 0)
    v2 = tf.Vertex.ByCoordinates(cx + hw, cy - hl, 0)
    v3 = tf.Vertex.ByCoordinates(cx + hw, cy + hl, 0)
    v4 = tf.Vertex.ByCoordinates(cx - hw, cy + hl, 0)
    
    e1 = tf.Edge.ByStartVertexEndVertex(v1, v2)
    e2 = tf.Edge.ByStartVertexEndVertex(v2, v3)
    e3 = tf.Edge.ByStartVertexEndVertex(v3, v4)
    e4 = tf.Edge.ByStartVertexEndVertex(v4, v1)
    
    return tf.Wire.ByEdges([e1, e2, e3, e4])


def create_l_shape(width, length, arm_width, cx=0, cy=0):
    """Create an L-shaped wire."""
    # L-shape vertices (6 points)
    v1 = tf.Vertex.ByCoordinates(cx, cy, 0)
    v2 = tf.Vertex.ByCoordinates(cx + width, cy, 0)
    v3 = tf.Vertex.ByCoordinates(cx + width, cy + arm_width, 0)
    v4 = tf.Vertex.ByCoordinates(cx + arm_width, cy + arm_width, 0)
    v5 = tf.Vertex.ByCoordinates(cx + arm_width, cy + length, 0)
    v6 = tf.Vertex.ByCoordinates(cx, cy + length, 0)
    
    edges = [
        tf.Edge.ByStartVertexEndVertex(v1, v2),
        tf.Edge.ByStartVertexEndVertex(v2, v3),
        tf.Edge.ByStartVertexEndVertex(v3, v4),
        tf.Edge.ByStartVertexEndVertex(v4, v5),
        tf.Edge.ByStartVertexEndVertex(v5, v6),
        tf.Edge.ByStartVertexEndVertex(v6, v1)
    ]
    
    return tf.Wire.ByEdges(edges)


def create_t_shape(width, length, stem_width, cx=0, cy=0):
    """Create a T-shaped wire."""
    # T-shape vertices (8 points)
    hw = width / 2
    sw = stem_width / 2
    
    v1 = tf.Vertex.ByCoordinates(cx - hw, cy + length - stem_width, 0)  # top-left
    v2 = tf.Vertex.ByCoordinates(cx + hw, cy + length - stem_width, 0)  # top-right
    v3 = tf.Vertex.ByCoordinates(cx + hw, cy + length, 0)               # top-right corner
    v4 = tf.Vertex.ByCoordinates(cx + sw, cy + length, 0)               # inner top-right
    v5 = tf.Vertex.ByCoordinates(cx + sw, cy, 0)                        # bottom-right
    v6 = tf.Vertex.ByCoordinates(cx - sw, cy, 0)                        # bottom-left
    v7 = tf.Vertex.ByCoordinates(cx - sw, cy + length, 0)               # inner top-left
    v8 = tf.Vertex.ByCoordinates(cx - hw, cy + length, 0)               # top-left corner
    
    edges = [
        tf.Edge.ByStartVertexEndVertex(v1, v2),
        tf.Edge.ByStartVertexEndVertex(v2, v3),
        tf.Edge.ByStartVertexEndVertex(v3, v4),
        tf.Edge.ByStartVertexEndVertex(v4, v5),
        tf.Edge.ByStartVertexEndVertex(v5, v6),
        tf.Edge.ByStartVertexEndVertex(v6, v7),
        tf.Edge.ByStartVertexEndVertex(v7, v8),
        tf.Edge.ByStartVertexEndVertex(v8, v1)
    ]
    
    return tf.Wire.ByEdges(edges)


# Create some test shapes
rect1 = create_rectangle(2, 2)
rect2 = create_rectangle(4, 2)
l_shape = create_l_shape(2, 3, 1)
t_shape = create_t_shape(3, 2, 1)

print("Test shapes created:")
print(f"  Rectangle 2x2: {len(rect1.Edges())} edges")
print(f"  Rectangle 4x2: {len(rect2.Edges())} edges")
print(f"  L-Shape: {len(l_shape.Edges())} edges")
print(f"  T-Shape: {len(t_shape.Edges())} edges")

## 3. Visualize Shapes

In [ ]:
def plot_wire(wire, fig, color='blue', name='Wire', row=1, col=1):
    """Plot a wire on a plotly figure."""
    edges = wire.Edges()
    
    for i, edge in enumerate(edges):
        verts = edge.Vertices()
        if len(verts) == 2:
            p1 = verts[0].Coordinates()
            p2 = verts[1].Coordinates()
            fig.add_trace(go.Scatter(
                x=[p1[0], p2[0]],
                y=[p1[1], p2[1]],
                mode='lines',
                line=dict(color=color, width=3),
                showlegend=(i == 0),
                name=name
            ), row=row, col=col)
    
    # Add vertices
    vertices = wire.Vertices()
    fig.add_trace(go.Scatter(
        x=[v.X() for v in vertices],
        y=[v.Y() for v in vertices],
        mode='markers',
        marker=dict(size=8, color=color),
        showlegend=False
    ), row=row, col=col)


# Visualize shapes
fig_shapes = make_subplots(
    rows=2, cols=2,
    subplot_titles=['Rectangle 2x2', 'Rectangle 4x2', 'L-Shape', 'T-Shape']
)

plot_wire(rect1, fig_shapes, 'blue', 'Rectangle', 1, 1)
plot_wire(rect2, fig_shapes, 'green', 'Wide Rectangle', 1, 2)
plot_wire(l_shape, fig_shapes, 'orange', 'L-Shape', 2, 1)
plot_wire(t_shape, fig_shapes, 'purple', 'T-Shape', 2, 2)

fig_shapes.update_xaxes(scaleanchor="y", scaleratio=1)
fig_shapes.update_layout(
    title='Shape Patterns for Grammar Rules',
    height=600,
    width=800
)

fig_shapes.show()

## 4. Define Grammar Rules

In [ ]:
# Define some architectural grammar rules

# Rule 1: Split a rectangle horizontally into two rooms
def create_split_rule():
    """Rule: Split rectangle into two rooms."""
    # LHS: Single rectangle
    lhs = create_rectangle(2, 2, cx=-3)
    
    # RHS: Two adjacent rectangles
    v1 = tf.Vertex.ByCoordinates(2, -1, 0)
    v2 = tf.Vertex.ByCoordinates(4, -1, 0)
    v3 = tf.Vertex.ByCoordinates(4, 1, 0)
    v4 = tf.Vertex.ByCoordinates(2, 1, 0)
    v5 = tf.Vertex.ByCoordinates(3, -1, 0)  # Split point
    v6 = tf.Vertex.ByCoordinates(3, 1, 0)   # Split point
    
    edges = [
        tf.Edge.ByStartVertexEndVertex(v1, v5),
        tf.Edge.ByStartVertexEndVertex(v5, v2),
        tf.Edge.ByStartVertexEndVertex(v2, v3),
        tf.Edge.ByStartVertexEndVertex(v3, v6),
        tf.Edge.ByStartVertexEndVertex(v6, v4),
        tf.Edge.ByStartVertexEndVertex(v4, v1),
        tf.Edge.ByStartVertexEndVertex(v5, v6)  # Internal wall
    ]
    
    rhs = tf.Wire.ByEdges(edges[:6])  # Outer boundary only for Wire
    
    return GrammarRule("split_h", lhs, rhs, "Split rectangle horizontally")


# Rule 2: Add corridor to a room
def create_corridor_rule():
    """Rule: Add corridor extension."""
    # LHS: Rectangle
    lhs = create_rectangle(2, 2, cx=-3)
    
    # RHS: Rectangle with corridor
    rhs = create_l_shape(3, 3, 1, cx=2, cy=-1)
    
    return GrammarRule("add_corridor", lhs, rhs, "Add corridor to room")


# Rule 3: Transform L to T
def create_l_to_t_rule():
    """Rule: Transform L-shape to T-shape."""
    lhs = create_l_shape(2, 3, 1, cx=-4, cy=-1.5)
    rhs = create_t_shape(3, 2, 1, cx=2, cy=-1)
    
    return GrammarRule("l_to_t", lhs, rhs, "Transform L-shape to T-shape")


# Create rules
rules = [
    create_split_rule(),
    create_corridor_rule(),
    create_l_to_t_rule()
]

print("Grammar Rules Created:")
for i, rule in enumerate(rules, 1):
    print(f"  {i}. {rule.name}: {rule.description}")

## 5. Visualize Grammar Rules

In [ ]:
def visualize_rule(rule, row_offset=0):
    """Create a visualization of a grammar rule."""
    fig = go.Figure()
    
    # Plot LHS
    lhs_edges = rule.lhs_wire.Edges()
    for edge in lhs_edges:
        verts = edge.Vertices()
        if len(verts) == 2:
            p1 = verts[0].Coordinates()
            p2 = verts[1].Coordinates()
            fig.add_trace(go.Scatter(
                x=[p1[0], p2[0]], y=[p1[1], p2[1]],
                mode='lines',
                line=dict(color='blue', width=3),
                showlegend=False
            ))
    
    # Plot RHS
    rhs_edges = rule.rhs_wire.Edges()
    for edge in rhs_edges:
        verts = edge.Vertices()
        if len(verts) == 2:
            p1 = verts[0].Coordinates()
            p2 = verts[1].Coordinates()
            fig.add_trace(go.Scatter(
                x=[p1[0], p2[0]], y=[p1[1], p2[1]],
                mode='lines',
                line=dict(color='green', width=3),
                showlegend=False
            ))
    
    # Add arrow
    fig.add_annotation(
        x=0, y=0,
        ax=-0.8, ay=0,
        xref='x', yref='y',
        axref='x', ayref='y',
        showarrow=True,
        arrowhead=2,
        arrowsize=2,
        arrowwidth=3,
        arrowcolor='red'
    )
    
    # Add labels
    fig.add_annotation(x=-3, y=2.5, text="LHS", showarrow=False, font=dict(size=14, color='blue'))
    fig.add_annotation(x=3, y=2.5, text="RHS", showarrow=False, font=dict(size=14, color='green'))
    
    fig.update_layout(
        title=f"Rule: {rule.name} - {rule.description}",
        xaxis=dict(scaleanchor='y', scaleratio=1, range=[-6, 7]),
        yaxis=dict(range=[-3, 4]),
        width=700, height=400
    )
    
    return fig


# Show each rule
for rule in rules:
    fig = visualize_rule(rule)
    fig.show()

## 6. Grammar Rule Application

We can apply grammar rules through explicit transformations. topologic_fast provides:
- `Wire.IsSimilar(wire_a, wire_b, ang_tolerance, tolerance)` - Check if wires have similar shapes
- Manual cell splitting and transformation functions

Let's demonstrate pattern matching with `Wire.IsSimilar()` and then show rule application.

In [ ]:
# Demonstrate Wire.IsSimilar() for pattern matching
print("=== Wire Similarity Testing ===\n")

# Create test wires
rect_small = create_rectangle(1, 1)
rect_large = create_rectangle(4, 4)  # Same proportions, different size
rect_wide = create_rectangle(4, 2)   # Different proportions

# Test similarity
print("Testing wire similarity (angle-based):")
print(f"  Small rect vs Large rect (same proportions): {tf.Wire.IsSimilar(rect_small, rect_large, 0.1)}")
print(f"  Small rect vs Wide rect (different proportions): {tf.Wire.IsSimilar(rect_small, rect_wide, 0.1)}")
print(f"  L-shape vs T-shape: {tf.Wire.IsSimilar(l_shape, t_shape, 0.1)}")

# Create two similar L-shapes at different scales
l_shape_small = create_l_shape(1, 1.5, 0.5)
l_shape_big = create_l_shape(3, 4.5, 1.5)
print(f"  L-shape small vs L-shape big (same proportions): {tf.Wire.IsSimilar(l_shape_small, l_shape_big, 0.1)}")

print("\n=== Rule Application Demo ===\n")

def apply_split_rule(cell, split_ratio=0.5):
    """
    Apply the split rule to a cell.
    
    This is a simplified implementation that splits a box cell horizontally.
    """
    # Get the bounding box of the cell
    faces = cell.Faces()
    
    # Find min/max coordinates
    all_verts = []
    for face in faces:
        all_verts.extend(face.Vertices())
    
    x_coords = [v.X() for v in all_verts]
    y_coords = [v.Y() for v in all_verts]
    z_coords = [v.Z() for v in all_verts]
    
    x_min, x_max = min(x_coords), max(x_coords)
    y_min, y_max = min(y_coords), max(y_coords)
    z_min, z_max = min(z_coords), max(z_coords)
    
    # Split point
    x_split = x_min + (x_max - x_min) * split_ratio
    
    # Create two new cells
    cell1 = tf.Cell.Box(x_min, y_min, z_min, 
                        x_split - x_min, y_max - y_min, z_max - z_min)
    cell2 = tf.Cell.Box(x_split, y_min, z_min,
                        x_max - x_split, y_max - y_min, z_max - z_min)
    
    return [cell1, cell2]


# Example: Apply split rule to a cell
original_cell = tf.Cell.Box(0, 0, 0, 4, 3, 2)
print(f"Original cell volume: {original_cell.Volume():.2f}")

result_cells = apply_split_rule(original_cell, 0.4)
print(f"\nAfter applying split rule:")
for i, cell in enumerate(result_cells):
    print(f"  Cell {i+1} volume: {cell.Volume():.2f}")

print(f"\nTotal volume preserved: {sum(c.Volume() for c in result_cells):.2f}")

In [ ]:
# Visualize the split rule application
fig_split = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Before: Single Room', 'After: Two Rooms'],
    specs=[[{'type': 'scene'}, {'type': 'scene'}]]
)

def add_cell_to_fig(cell, fig, color, name, row, col, opacity=0.5):
    """Add a cell's faces to a 3D figure."""
    faces = cell.Faces()
    for i, face in enumerate(faces):
        vertices = face.Vertices()
        coords = [v.Coordinates() for v in vertices]
        
        x = [c[0] for c in coords]
        y = [c[1] for c in coords]
        z = [c[2] for c in coords]
        
        fig.add_trace(go.Mesh3d(
            x=x, y=y, z=z,
            color=color,
            opacity=opacity,
            alphahull=0,
            name=name,
            showlegend=(i == 0)
        ), row=row, col=col)

# Original cell
add_cell_to_fig(original_cell, fig_split, 'blue', 'Original', 1, 1, 0.6)

# Result cells
add_cell_to_fig(result_cells[0], fig_split, 'green', 'Room 1', 1, 2, 0.6)
add_cell_to_fig(result_cells[1], fig_split, 'orange', 'Room 2', 1, 2, 0.6)

fig_split.update_layout(
    title='Split Rule Application',
    height=500,
    width=1000
)

fig_split.show()

## 7. Recursive Rule Application

Apply grammar rules recursively to generate complex layouts.

In [ ]:
def recursive_split(cell, depth, direction='x'):
    """
    Recursively apply the split rule to create a subdivision.
    
    Args:
        cell: The cell to split
        depth: How many levels of recursion
        direction: 'x' or 'y' for split direction
    """
    if depth == 0:
        return [cell]
    
    # Get bounds
    faces = cell.Faces()
    all_verts = []
    for face in faces:
        all_verts.extend(face.Vertices())
    
    x_coords = [v.X() for v in all_verts]
    y_coords = [v.Y() for v in all_verts]
    z_coords = [v.Z() for v in all_verts]
    
    x_min, x_max = min(x_coords), max(x_coords)
    y_min, y_max = min(y_coords), max(y_coords)
    z_min, z_max = min(z_coords), max(z_coords)
    
    width = x_max - x_min
    length = y_max - y_min
    height = z_max - z_min
    
    # Split based on direction
    if direction == 'x':
        cell1 = tf.Cell.Box(x_min, y_min, z_min, width/2, length, height)
        cell2 = tf.Cell.Box(x_min + width/2, y_min, z_min, width/2, length, height)
        next_dir = 'y'
    else:
        cell1 = tf.Cell.Box(x_min, y_min, z_min, width, length/2, height)
        cell2 = tf.Cell.Box(x_min, y_min + length/2, z_min, width, length/2, height)
        next_dir = 'x'
    
    # Recurse
    result = []
    result.extend(recursive_split(cell1, depth - 1, next_dir))
    result.extend(recursive_split(cell2, depth - 1, next_dir))
    
    return result


# Apply recursive split
base_cell = tf.Cell.Box(0, 0, 0, 8, 8, 3)
subdivided_cells = recursive_split(base_cell, depth=3)

print(f"Started with 1 cell, ended with {len(subdivided_cells)} cells")
print(f"Total volume: {sum(c.Volume() for c in subdivided_cells):.2f}")

In [ ]:
# Visualize the recursive subdivision
import random

# Color palette
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7', '#DDA0DD', '#98D8C8', '#F7DC6F']

fig_recursive = go.Figure()

for i, cell in enumerate(subdivided_cells):
    color = colors[i % len(colors)]
    faces = cell.Faces()
    
    for j, face in enumerate(faces):
        vertices = face.Vertices()
        coords = [v.Coordinates() for v in vertices]
        
        x = [c[0] for c in coords]
        y = [c[1] for c in coords]
        z = [c[2] for c in coords]
        
        fig_recursive.add_trace(go.Mesh3d(
            x=x, y=y, z=z,
            color=color,
            opacity=0.7,
            alphahull=0,
            name=f'Room {i+1}',
            showlegend=(j == 0)
        ))

fig_recursive.update_layout(
    title=f'Recursive Split Grammar: {len(subdivided_cells)} Rooms Generated',
    scene=dict(
        aspectmode='data',
        camera=dict(eye=dict(x=1.5, y=1.5, z=1))
    ),
    width=800,
    height=600
)

fig_recursive.show()

## 8. Create Connectivity Graph from Generated Layout

In [ ]:
# Create CellComplex from subdivided cells
layout_complex = tf.CellComplex.ByCells(subdivided_cells)

# Create dual graph
layout_graph = tf.Graph.ByTopology(layout_complex)

print(f"Generated Layout Statistics:")
print(f"  Rooms (cells):      {len(subdivided_cells)}")
print(f"  Graph vertices:     {layout_graph.Order()}")
print(f"  Connections:        {layout_graph.Size()}")
print(f"  Graph density:      {layout_graph.Density():.3f}")
print(f"  Diameter:           {layout_graph.Diameter()}")

In [ ]:
# Visualize layout with connectivity graph
fig_layout = go.Figure()

# Draw rooms (2D projection)
for i, cell in enumerate(subdivided_cells):
    color = colors[i % len(colors)]
    faces = cell.Faces()
    
    for face in faces:
        vertices = face.Vertices()
        coords = [v.Coordinates() for v in vertices]
        z_coords = [c[2] for c in coords]
        
        # Bottom face (z = 0)
        if all(abs(z) < 0.01 for z in z_coords):
            x = [c[0] for c in coords] + [coords[0][0]]
            y = [c[1] for c in coords] + [coords[0][1]]
            
            fig_layout.add_trace(go.Scatter(
                x=x, y=y,
                fill='toself',
                fillcolor=color,
                line=dict(color='black', width=2),
                name=f'Room {i+1}',
                showlegend=True
            ))
            break

# Draw graph edges
graph_edges = layout_graph.Edges()
for edge in graph_edges:
    verts = edge.Vertices()
    if len(verts) == 2:
        p1 = verts[0].Coordinates()
        p2 = verts[1].Coordinates()
        fig_layout.add_trace(go.Scatter(
            x=[p1[0], p2[0]], y=[p1[1], p2[1]],
            mode='lines',
            line=dict(color='red', width=3),
            showlegend=False
        ))

# Draw graph vertices
graph_verts = layout_graph.Vertices()
fig_layout.add_trace(go.Scatter(
    x=[v.X() for v in graph_verts],
    y=[v.Y() for v in graph_verts],
    mode='markers',
    marker=dict(size=12, color='red', line=dict(color='darkred', width=2)),
    name='Graph Nodes'
))

fig_layout.update_layout(
    title='Generated Floor Plan with Connectivity Graph',
    xaxis=dict(title='X', scaleanchor='y', scaleratio=1),
    yaxis=dict(title='Y'),
    width=700,
    height=700
)

fig_layout.show()

## Summary

This notebook demonstrated graph grammar concepts using topologic_fast:

### Key Concepts

1. **Grammar Rules** consist of:
   - Left-hand side (LHS): The pattern to match
   - Right-hand side (RHS): The replacement pattern
   - Can be stored using topological structures (Wires, Edges)

2. **Rule Application**:
   - Pattern matching with `Wire.IsSimilar()`
   - Transformation and replacement
   - Recursive application for complex layouts

3. **Practical Applications**:
   - Floor plan generation
   - Design space exploration
   - Encoding architectural rules

### topologic_fast Features Used

- `tf.Vertex.ByCoordinates()` - Create vertices
- `tf.Edge.ByStartVertexEndVertex()` - Create edges
- `tf.Wire.ByEdges()` - Create wires from edges
- `tf.Wire.IsSimilar()` - Check if two wires have similar shapes
- `tf.Cell.Box()` - Create box cells
- `tf.CellComplex.ByCells()` - Combine cells
- `tf.Graph.ByTopology()` - Create connectivity graph

### Features Not Yet Available (from topologicpy)

- `Topology.IsSimilar()` - General topology comparison with transformation matrix
- `ShapeGrammar` class - Full rule management with operations (Replace, Transform, Union, etc.)

To achieve full parity with topologicpy's shape grammar system, these features would need to be implemented.